# 📈 Linear Regression — Practice Notebook

**Difficulty**: ⭐ Beginner  
**Time**: ~45 minutes  
**Prerequisites**: Basic linear algebra, calculus, Python/NumPy

---

## What You'll Learn
- How linear regression works (math + intuition)
- Implement gradient descent from scratch
- Implement the normal equation
- Use scikit-learn for linear regression
- Evaluate regression models
- Common interview questions & answers

---
## 🎯 Section 1: Overview

**Linear Regression** is one of the simplest and most fundamental ML algorithms. It models the relationship between a dependent variable (target) and one or more independent variables (features) by fitting a **linear equation** to the data.

### When to Use
- Predicting continuous values (house prices, temperature, sales)
- Understanding relationships between variables
- When the relationship between features and target is approximately linear

### Types
- **Simple Linear Regression**: One feature → `y = θ₀ + θ₁x`
- **Multiple Linear Regression**: Multiple features → `y = θ₀ + θ₁x₁ + θ₂x₂ + ... + θₙxₙ`

### Key Assumptions (LINE)
1. **L**inearity — The relationship between X and y is linear
2. **I**ndependence — Observations are independent of each other
3. **N**ormality — Residuals are normally distributed
4. **E**qual variance (Homoscedasticity) — Residuals have constant variance

---
## 📐 Section 2: Math & Intuition

### Hypothesis Function
$$h_\theta(x) = \theta^T x = \theta_0 + \theta_1 x_1 + \theta_2 x_2 + \cdots + \theta_n x_n$$

In matrix form: $\hat{y} = X\theta$

### Cost Function (Mean Squared Error)
$$J(\theta) = \frac{1}{2m} \sum_{i=1}^{m} (h_\theta(x^{(i)}) - y^{(i)})^2$$

The factor of `1/2` is for convenience when taking derivatives.

### Gradient Descent Update Rule
$$\theta_j := \theta_j - \alpha \frac{\partial J}{\partial \theta_j}$$

Where:
$$\frac{\partial J}{\partial \theta_j} = \frac{1}{m} \sum_{i=1}^{m} (h_\theta(x^{(i)}) - y^{(i)}) \cdot x_j^{(i)}$$

In vectorized form:
$$\theta := \theta - \frac{\alpha}{m} X^T(X\theta - y)$$

### Normal Equation (Closed-Form Solution)
$$\theta = (X^T X)^{-1} X^T y$$

| Gradient Descent | Normal Equation |
|---|---|
| Need to choose α | No learning rate |
| Many iterations | One computation |
| O(kn²) | O(n³) for matrix inverse |
| Works well with large n | Slow when n > 10,000 features |

---
## 🔧 Section 3: Implementation from Scratch

Let's implement linear regression step by step using only NumPy.

In [ ]:
# Imports (pre-filled)
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed for reproducibility
np.random.seed(42)

# Plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('Setup complete! ✅')

### 3.1 Generate Synthetic Data

We'll create a dataset where `y = 4 + 3x + noise`

In [ ]:
# Generate synthetic data: y = 4 + 3x + noise
m = 100  # number of samples
X = 2 * np.random.rand(m, 1)  # features in [0, 2]
y = 4 + 3 * X + np.random.randn(m, 1) * 0.5  # true relationship + noise

# Visualize the data
plt.figure(figsize=(10, 6))
plt.scatter(X, y, alpha=0.7, edgecolors='k', linewidth=0.5)
plt.xlabel('X (Feature)', fontsize=12)
plt.ylabel('y (Target)', fontsize=12)
plt.title('Synthetic Linear Data', fontsize=14)
plt.show()

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'True parameters: θ₀ = 4, θ₁ = 3')

### 3.2 Add Bias Term

To include the intercept θ₀, we add a column of 1s to X.

In [ ]:
# TODO: Add a column of ones to X for the bias/intercept term
# Hint: Use np.c_ or np.hstack to concatenate a column of ones
# X_b should have shape (100, 2) — first column all 1s, second column is X

X_b = None  # TODO: Replace with your implementation

# Verify
assert X_b is not None, "Don't forget to implement this!"
assert X_b.shape == (100, 2), f"Expected shape (100, 2), got {X_b.shape}"
assert np.all(X_b[:, 0] == 1), "First column should be all ones"
print(f'X_b shape: {X_b.shape} ✅')
print(f'First 3 rows:\n{X_b[:3]}')

### 3.3 Normal Equation

Implement the closed-form solution: $\theta = (X^T X)^{-1} X^T y$

In [ ]:
def normal_equation(X, y):
    """
    Compute the optimal theta using the normal equation.
    
    Parameters:
    -----------
    X : np.ndarray of shape (m, n+1) — design matrix with bias column
    y : np.ndarray of shape (m, 1)   — target values
    
    Returns:
    --------
    theta : np.ndarray of shape (n+1, 1) — optimal parameters
    """
    # TODO: Implement the normal equation
    # Hint: Use np.linalg.inv() for matrix inverse and @ or np.dot() for multiplication
    # Formula: theta = (X^T @ X)^(-1) @ X^T @ y
    
    theta = None  # TODO: Replace with your implementation
    
    return theta


# Test your implementation
theta_ne = normal_equation(X_b, y)
assert theta_ne is not None, "Don't forget to implement the normal equation!"
print(f'Normal Equation Result:')
print(f'  θ₀ (intercept) = {theta_ne[0, 0]:.4f}  (expected ≈ 4.0)')
print(f'  θ₁ (slope)     = {theta_ne[1, 0]:.4f}  (expected ≈ 3.0)')

# Soft check — should be close to true values
assert abs(theta_ne[0, 0] - 4.0) < 1.0, "θ₀ seems too far from expected value"
assert abs(theta_ne[1, 0] - 3.0) < 1.0, "θ₁ seems too far from expected value"
print('\n✅ Normal equation looks correct!')

### 3.4 Gradient Descent

Now implement the iterative approach. This is the more commonly asked version in interviews!

In [ ]:
def compute_cost(X, y, theta):
    """
    Compute the MSE cost function J(θ).
    
    Parameters:
    -----------
    X     : np.ndarray of shape (m, n+1)
    y     : np.ndarray of shape (m, 1)
    theta : np.ndarray of shape (n+1, 1)
    
    Returns:
    --------
    cost : float — the MSE cost
    """
    m = len(y)
    
    # TODO: Compute the cost J(θ) = (1/2m) * sum((Xθ - y)²)
    # Step 1: Compute predictions (X @ theta)
    # Step 2: Compute errors (predictions - y)
    # Step 3: Compute cost = (1/(2m)) * sum(errors²)
    
    cost = None  # TODO: Replace
    
    return cost


# Quick test
theta_test = np.array([[0], [0]])
test_cost = compute_cost(X_b, y, theta_test)
assert test_cost is not None, "Implement compute_cost!"
print(f'Cost with θ=[0,0]: {test_cost:.4f}')
print(f'Cost with θ from normal eq: {compute_cost(X_b, y, theta_ne):.4f} (should be much lower)')

In [ ]:
def gradient_descent(X, y, theta, alpha, num_iters):
    """
    Perform batch gradient descent to learn theta.
    
    Parameters:
    -----------
    X         : np.ndarray of shape (m, n+1)
    y         : np.ndarray of shape (m, 1)
    theta     : np.ndarray of shape (n+1, 1) — initial parameters
    alpha     : float — learning rate
    num_iters : int — number of iterations
    
    Returns:
    --------
    theta   : np.ndarray — learned parameters
    history : list — cost at each iteration (for plotting convergence)
    """
    m = len(y)
    history = []
    
    for i in range(num_iters):
        # TODO: Implement one step of gradient descent
        # Step 1: Compute predictions = X @ theta
        # Step 2: Compute errors = predictions - y
        # Step 3: Compute gradients = (1/m) * X^T @ errors
        # Step 4: Update theta = theta - alpha * gradients
        
        pass  # TODO: Replace with your implementation
        
        # Record cost for convergence plot
        history.append(compute_cost(X, y, theta))
    
    return theta, history


# Run gradient descent
theta_init = np.zeros((2, 1))
alpha = 0.1       # learning rate
num_iters = 1000   # iterations

theta_gd, cost_history = gradient_descent(X_b, y, theta_init.copy(), alpha, num_iters)

print(f'Gradient Descent Result:')
print(f'  θ₀ (intercept) = {theta_gd[0, 0]:.4f}  (expected ≈ 4.0)')
print(f'  θ₁ (slope)     = {theta_gd[1, 0]:.4f}  (expected ≈ 3.0)')
print(f'  Final cost      = {cost_history[-1]:.4f}')

### 3.5 Visualize Convergence

In [ ]:
# TODO: Plot the cost history to see gradient descent convergence
# Hint: plt.plot(cost_history)
# Label axes: 'Iteration' and 'Cost J(θ)'
# Title: 'Gradient Descent Convergence'

# YOUR CODE HERE


### 3.6 Visualize the Fit

In [ ]:
# TODO: Plot the data points and the regression line from gradient descent
# Hint:
#   1. Scatter plot the original X, y data
#   2. Create X_plot = np.linspace(0, 2, 100).reshape(-1, 1)
#   3. Add bias column to X_plot
#   4. Compute predictions: y_plot = X_plot_b @ theta_gd
#   5. Plot the line in red

# YOUR CODE HERE


---
## 📦 Section 4: Using scikit-learn

Now let's do the same thing with scikit-learn in just a few lines.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# TODO: Implement linear regression using scikit-learn
# Step 1: Split data into train/test sets (80/20)
# Step 2: Create a LinearRegression model
# Step 3: Fit on training data
# Step 4: Predict on test data
# Step 5: Print coefficients and intercept
# Step 6: Compute and print MSE, RMSE, MAE, R² on test set

# Hint: X here is the original X without bias column (sklearn adds it)

# YOUR CODE HERE


### 4.1 Multiple Linear Regression with scikit-learn

Let's try with multiple features.

In [ ]:
# Generate multi-feature data: y = 2 + 3x₁ - 1.5x₂ + 0.5x₃ + noise
np.random.seed(42)
m_multi = 200
X_multi = np.random.randn(m_multi, 3)
y_multi = 2 + 3 * X_multi[:, 0] - 1.5 * X_multi[:, 1] + 0.5 * X_multi[:, 2] + np.random.randn(m_multi) * 0.3

# TODO: Fit a LinearRegression model to X_multi, y_multi
# Print the intercept and coefficients
# Compare with true values: intercept=2, coefs=[3, -1.5, 0.5]

# YOUR CODE HERE


---
## 🧪 Section 5: Experiments

### 5.1 Effect of Learning Rate

In [ ]:
# TODO: Run gradient descent with 3 different learning rates: 0.005, 0.1, 0.5
# Plot the convergence curves on the same graph
# Observe: too small = slow, too large = oscillates/diverges, just right = fast convergence

# Hint:
# alphas = [0.005, 0.1, 0.5]
# For each alpha, run gradient_descent() and plot cost_history
# Use plt.legend() to label each curve

# YOUR CODE HERE


### 5.2 Feature Scaling Impact

Feature scaling is crucial for gradient descent convergence. Let's see why.

In [ ]:
# Create data with features on VERY different scales
np.random.seed(42)
X_unscaled = np.column_stack([
    np.random.randn(100) * 1000,   # Feature 1: range ~[-3000, 3000]
    np.random.randn(100) * 0.001   # Feature 2: range ~[-0.003, 0.003]
])
y_unscaled = 5 + 2 * X_unscaled[:, 0] + 3000 * X_unscaled[:, 1] + np.random.randn(100) * 10

# TODO: 
# 1. Try fitting gradient descent WITHOUT scaling — observe the cost history
# 2. Apply StandardScaler (or manual z-score normalization) to X_unscaled
# 3. Try fitting gradient descent WITH scaling — compare convergence
# 4. Plot both convergence curves side by side

# Hint for manual scaling: X_scaled = (X - X.mean(axis=0)) / X.std(axis=0)

# YOUR CODE HERE


### 5.3 Residual Analysis

Good practice: always check residuals to validate assumptions.

In [ ]:
# TODO: For the simple linear regression model (from sklearn):
# 1. Compute residuals = y_true - y_predicted
# 2. Create a 2x2 subplot:
#    (a) Residuals vs Fitted values — should show no pattern
#    (b) Histogram of residuals — should be roughly normal
#    (c) Q-Q plot — from scipy.stats.probplot()
#    (d) Residuals vs Feature value

# YOUR CODE HERE


---
## ❓ Section 6: Interview Questions

Try to answer these **before** looking at the answers below.

### Q1: What are the assumptions of linear regression?
<details><summary>Click for answer</summary>

**LINE:**
1. **Linearity** — The relationship between X and y is linear
2. **Independence** — Residuals are independent (no autocorrelation)
3. **Normality** — Residuals are normally distributed
4. **Equal variance** (Homoscedasticity) — Residuals have constant variance across all levels of X

Additional: No multicollinearity (for multiple regression)
</details>

### Q2: What is multicollinearity? How do you detect and handle it?
<details><summary>Click for answer</summary>

**Multicollinearity** is when two or more independent variables are highly correlated with each other.

**Detection:**
- Correlation matrix / heatmap (|r| > 0.8 is concerning)
- Variance Inflation Factor (VIF > 5-10 indicates multicollinearity)

**Handling:**
- Remove one of the correlated features
- Use PCA to combine correlated features
- Use regularization (Ridge regression handles it well)
</details>

### Q3: Difference between R² and Adjusted R²?
<details><summary>Click for answer</summary>

- **R²** measures the proportion of variance explained. It always increases (or stays same) when adding more features, even irrelevant ones.
- **Adjusted R²** penalizes for the number of features: `1 - (1-R²)(n-1)/(n-p-1)`. It can decrease if an added feature doesn't improve the model enough.
- Use **Adjusted R²** when comparing models with different numbers of features.
</details>

### Q4: When would you use gradient descent over the normal equation?
<details><summary>Click for answer</summary>

- **Normal equation**: O(n³) due to matrix inversion. Best when n (features) < 10,000.
- **Gradient descent**: O(kn²) per iteration. Preferred when n is very large.
- Gradient descent also works for models beyond linear regression (logistic, neural nets, etc.)
- Normal equation can have numerical issues if X^T X is not invertible (use pseudoinverse).
</details>

### Q5: What is the difference between MSE, RMSE, and MAE?
<details><summary>Click for answer</summary>

| Metric | Formula | Properties |
|---|---|---|
| MSE | `mean((y-ŷ)²)` | Heavily penalizes large errors, not in original units |
| RMSE | `sqrt(MSE)` | Same units as target, penalizes large errors |
| MAE | `mean(|y-ŷ|)` | Same units, more robust to outliers |

Use RMSE when large errors are particularly bad. Use MAE when you want robustness to outliers.
</details>

### Q6: Can linear regression be used for classification?
<details><summary>Click for answer</summary>

Technically yes, but it's a poor choice because:
- Output is unbounded (can predict < 0 or > 1 for probabilities)
- Sensitive to outliers that shift the decision boundary
- Assumes equal variance of errors across classes
- Use **logistic regression** instead for binary classification
</details>

### Q7: What happens if features are not scaled before gradient descent?
<details><summary>Click for answer</summary>

The cost function contours become elongated ellipses instead of circles. This causes gradient descent to oscillate and converge very slowly (or diverge). Feature scaling (standardization or normalization) makes the contours more circular, enabling faster convergence.

Note: The **normal equation** does NOT require feature scaling.
</details>

---
## 🏆 Section 7: Challenge

### Challenge: Boston Housing (Synthetic Version)

Build a complete linear regression pipeline from scratch:

In [ ]:
# Synthetic housing dataset
np.random.seed(42)
n_houses = 500

# Features: sqft, bedrooms, age, distance_to_city
sqft = np.random.uniform(500, 4000, n_houses)
bedrooms = np.random.randint(1, 6, n_houses).astype(float)
age = np.random.uniform(0, 50, n_houses)
distance = np.random.uniform(1, 30, n_houses)

# True relationship
price = (100 * sqft + 15000 * bedrooms - 2000 * age - 5000 * distance 
         + 50000 + np.random.randn(n_houses) * 20000)

X_housing = np.column_stack([sqft, bedrooms, age, distance])
y_housing = price

print(f'Features shape: {X_housing.shape}')
print(f'Target shape: {y_housing.shape}')
print(f'Price range: ${y_housing.min():,.0f} — ${y_housing.max():,.0f}')

In [ ]:
# TODO: Complete challenge
# 1. Split into 80/20 train/test
# 2. Apply feature scaling (StandardScaler or manual)
# 3. Add bias column
# 4. Run gradient descent with appropriate learning rate
# 5. Also fit with sklearn LinearRegression for comparison
# 6. Report MSE, RMSE, R² for both methods
# 7. Plot actual vs predicted prices
# 8. Perform residual analysis

# YOUR CODE HERE


---
## ✅ Summary

In this notebook you learned:
- [x] How linear regression works mathematically
- [x] Normal equation (closed-form solution)
- [x] Gradient descent (iterative optimization)
- [x] scikit-learn usage for quick prototyping
- [x] Importance of feature scaling
- [x] Evaluation metrics (MSE, RMSE, MAE, R²)
- [x] Residual analysis for assumption checking

**Next**: [Polynomial Regression](../02-polynomial-regression/) →